<a href="https://colab.research.google.com/github/simonicarlo/ReefScanner/blob/main/ReefScanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦈 ReefScanner v1 — Motion Triage for BRUV Video

ReefScanner scans long underwater videos from a **static, mounted camera (BRUV-style)** and flags candidate sightings of large marine megafauna (sharks, rays) for human review.

- **Recall-oriented:** it over-flags rather than misses. A false positive costs ~10s of attention; a missed animal is the expensive failure.
- **v1 is motion-only (no ML).** The goal is to cut a human's watch time from hours to minutes.
- **Resumable:** Colab disconnects after idle/~12h. Processing checkpoints per-video, so a restart skips already-completed videos.
- **A labeling-data factory:** every event (clip + peak frame + motion bbox) is a pre-labeled candidate for a future fine-tuned detector.

This notebook walks through the pipeline **one stage at a time**. Set everything in the **⚙️ VARS** section, then run the cells top to bottom.

## 🔌 Setup — install ReefScanner & mount Drive

Code lives on GitHub; footage and outputs live on Google Drive. This installs the package (pinned deps for reproducibility) and mounts your Drive. `detector=none` (v1) needs **no** torch/CUDA.

In [ ]:
# Install ReefScanner from GitHub (pinned core deps; no ML/torch needed for detector='none').
!pip install -q 'git+https://github.com/simonicarlo/reefscanner.git'

# For detector='marine' / 'generic' (the v2 YOLO path), also install the ML extra:
# !pip install -q 'reefscanner[ml] @ git+https://github.com/simonicarlo/reefscanner.git'
# ...and fetch SharkTrack's weights to Drive (see the detector-mode cell below).

# Colab ships ffmpeg already. If you ever run somewhere without it, uncomment:
# !pip install -q imageio-ffmpeg

import reefscanner
print('ReefScanner', reefscanner.__version__)

In [ ]:
# Mount Google Drive so we can read footage and write results to it.
from google.colab import drive
drive.mount('/content/drive')

## ⚙️ VARS — I/O paths & configuration

**Everything you tune lives here.** Set the input/output folders on Drive, then adjust the recall-favoring defaults if needed. Every field maps 1:1 to `ReefScannerConfig` (SPEC §4).

In [ ]:
from reefscanner import ReefScannerConfig

# --- I/O (point these at your Drive folders) -----------------------------
INPUT_FOLDER  = '/content/drive/MyDrive/ReefScanner/input'    # videos in
OUTPUT_FOLDER = '/content/drive/MyDrive/ReefScanner/output'   # results out

cfg = ReefScannerConfig(
    input_folder=INPUT_FOLDER,
    output_folder=OUTPUT_FOLDER,

    # --- Step 2: sampling / downscale ---
    frame_sample_fps=1.0,     # decode at 1 fps (main compute saving)
    detect_downscale=0.5,     # detection on half-res frames (clips stay full-res)

    # --- Step 3: motion gating (recall source when detector='none') ---
    warmup_frames=10,         # let the background model settle (skip these)
    min_blob_area=500,        # reject small baitfish (detection-res pixels)
    persistence_frames=2,     # must persist N sampled frames (reject flicker)
    bg_subtractor='MOG2',     # 'MOG2' or 'KNN'
    detect_shadows=True,      # suppress shadow blobs

    # --- Step 5: event aggregation ---
    merge_gap_seconds=3.0,    # merge candidates closer than this into one event
    min_event_seconds=1.0,    # drop events shorter than this

    # --- Step 6: clip output ---
    pad_seconds=2.0,          # padding before/after each event
    reencode_clips=False,     # False = fast stream-copy; True = accurate cuts

    # --- Step 4: detector / recall source ---
    # 'none'   = motion-only (v1, no ML deps)
    # 'marine' = YOLO scans every frame (v2): finds distant animals by
    #            appearance, not motion size. Recommended for shark/ray.
    # 'generic'= stock COCO YOLO objectness (experimental, expect poor recall)
    detector='none',
    detector_weights=None,        # for 'marine': path to a SharkTrack / fine-tuned .pt
    ml_confidence_threshold=0.2,  # recall-first: keep it low
    target_classes=None,          # e.g. ['elasmobranch'] to keep only sharks/rays
    use_sahi=False,               # True = tiled inference for distant/small animals
    motion_prefilter=False,       # True = skip dead-static frames in detector mode
)
cfg

### 🦈 Optional: detector mode (v2 — find distant sharks/rays by appearance)

Motion-only (`detector='none'`) can't tell a *near small fish* from a *far large shark* — in pixels they're identical — so it over-flags close fish and misses distant animals. The **`marine`** detector fixes this by running a YOLO model on every frame.

Recommended weights: **[SharkTrack](https://github.com/filippovarini/sharktrack)** (YOLOv8, single `elasmobranch` class, trained on real BRUVS, MIT-licensed). See [`docs/model-research.md`](https://github.com/simonicarlo/ReefScanner/blob/main/docs/model-research.md) for the full model comparison.

To use it: install the `ml` extra (see the setup cell), download SharkTrack's release weights to Drive, then in **VARS** set:

```python
detector='marine'
detector_weights='/content/drive/MyDrive/ReefScanner/weights/sharktrack.pt'
use_sahi=True          # tiled inference — big recall win on distant animals
target_classes=['elasmobranch']
```

> ⚠️ The `ml` extra pulls in **Ultralytics YOLO (AGPL-3.0)** — fine for research/internal use; check the license before distributing or hosting a service. `detector='none'` stays dependency-free.

## 🔍 Stage 1 — Discover

Recursively find video files under the input folder (configurable extensions). This is just a preview — the scan in Stage 2 discovers them again itself.

In [ ]:
from reefscanner import discover_videos

videos = discover_videos(cfg.input_folder, cfg.extensions)
print(f'Found {len(videos)} video(s):')
for v in videos:
    print('  ', v)

## 🐟 Stages 2–6 — Scan (sample → detect/motion → aggregate → clips/CSV)

`process_folder` runs the whole per-video pipeline and is **resumable** — it skips videos already marked complete in the manifest. For each video it:

- **Step 2 — Preprocess:** decodes at `frame_sample_fps`, downscales for detection.
- **Step 3/4 — Recall stage** (set by `detector` in VARS):
  - `none` → **motion gating** (background subtraction + size/persistence) is the recall source.
  - `marine` → a **YOLO detector scans every frame**, finding animals by appearance (catches distant sharks motion would miss); motion becomes an optional pre-skip.
- **Step 5 — Event aggregation:** merge nearby candidate frames into events; drop too-short ones.
- **Step 6 — Output:** one padded clip + a peak-frame thumbnail per event, plus an append-only row in the CSV.

Re-running after a disconnect is safe: finished videos are skipped, no duplicate rows.

In [ ]:
from reefscanner import process_folder

batch = process_folder(cfg)   # progress bar shown; skips completed videos
print()
print(f'Processed {batch.n_videos_processed} video(s); {batch.n_events} event(s) emitted.')
print('CSV:     ', batch.csv_path)
print('Manifest:', batch.manifest_path)

## 👀 Review — results CSV & thumbnails

One row per event. `label`, `species`, `notes` are **left blank for you to fill during triage** — that labeling is what turns this output into training data later (SPEC §10). The thumbnail shows the peak frame with the motion box drawn.

In [ ]:
import pandas as pd
df = pd.read_csv(batch.csv_path)
print(f'{len(df)} events')
df.head(20)

In [ ]:
# Preview the thumbnail (peak frame + motion bbox) for the first few events.
from IPython.display import Image, display
for _, row in df.head(3).iterrows():
    print(f"{row['event_id']}  |  {row['source_video']}  |  {row['start_seconds']}-{row['end_seconds']}s")
    display(Image(filename=row['thumbnail_path'], width=360))

## 🔁 Resumability

The **manifest** (`reefscanner_manifest.json` in the output folder) is the source of truth for what's done. A video is marked complete only after its clips, thumbnails, and CSV rows are all flushed. If Colab disconnects mid-batch, just **re-run the Stage 2 cell** — completed videos are skipped; an interrupted video restarts from frame 0 (no mid-video resume, by design).

## 📊 False-positive rate (the v2 go/no-go)

Once you've triaged — filling the `label` column with `TP`/`FP` — compute the false-positive rate on real footage. This number decides whether ML (v2) earns its place (SPEC §8.6, §10). If motion-only triage is usable, you may not need ML at all.

In [ ]:
from reefscanner import fp_report
report = fp_report(batch.csv_path)
print(report.summary())